# Лабораторная работа № 4 — задание по библиотеке NumPy

**Дисциплина:** Технологии программирования. **Вариант:** 6.

## Постановка задачи

С помощью библиотеки NumPy (без использования циклов!) создать матрицы и векторы:

- $M \in \mathbb{N}^{85 \times 85}$: на диагонали нечётные числа $3, 5, 7, \ldots, 171$, вне диагонали — двойки;
- $N \in \mathbb{N}^{85 \times 85}$: чётные строки — повторяющийся паттерн `4 4 5`, нечётные строки — паттерн `4 7`;
- $p = \{1 + p_i\}$, где $p_i \sim N(2, \sigma^2=4)$, длина 85;
- $q^T = \{N_{24,j} \odot N_{71,j}\}_{j=0}^{84}$ — поэлементное произведение 24-й и 71-й строк матрицы $N$.

Затем вычислить заданные выражения. По варианту 6 основным является **выражение 1** (нумерация `(6-1) mod 5 + 1 = 1`); для полноты отчёта вычислены все пять выражений. Обозначения: $\odot$ — поэлементное произведение, $Tr$ — след матрицы, $I[\cdot]$ — индикаторная функция, $\|u\|_p$ — Lp-норма.

**Технологии:** Python 3, NumPy, Jupyter Notebook.

## Создание объектов

In [1]:
import numpy as np

rng = np.random.default_rng(seed=6)  # фиксация для воспроизводимости

# M: везде 2, на диагонали 3, 5, ..., 171
M = np.full((85, 85), 2, dtype=float)
np.fill_diagonal(M, 3 + 2 * np.arange(85))

# N: чередование строк-паттернов '4 4 5' и '4 7'
row_a = np.resize(np.array([4, 4, 5], dtype=float), 85)
row_b = np.resize(np.array([4, 7], dtype=float), 85)
rows = np.arange(85)[:, None] % 2 == 0
N = np.where(rows, row_a, row_b)

# p = 1 + N(2, 4) (дисперсия 4 -> std = 2), длина 85
p = 1 + rng.normal(loc=2, scale=2, size=85)

# q = поэлементное произведение строк N[24] и N[71]
q = N[24] * N[71]

print('M shape:', M.shape, 'diag:', M.diagonal()[:5], '...',
      M.diagonal()[-1])
print('N shape:', N.shape, '\nrow0:', N[0][:9], '\nrow1:', N[1][:9])
print('p[:5]:', np.round(p[:5], 4))
print('q[:5]:', q[:5], ' len q:', len(q))

M shape: (85, 85) diag: [ 3.  5.  7.  9. 11.] ... 171.0
N shape: (85, 85) 
row0: [4. 4. 5. 4. 4. 5. 4. 4. 5.] 
row1: [4. 7. 4. 7. 4. 7. 4. 7. 4.]
p[:5]: [ 5.1062  6.553  -2.1066  2.7241  5.0274]
q[:5]: [16. 28. 20. 28. 16.]  len q: 85


## Выражение 1 (основное по варианту 6)

$f = Tr\,(NM)^{p}$, $g = \sum_{i=0}^{84} I[i \bmod 2 = 0]\,(2p \odot q)_i$

Трактовка: $(NM)^p$ — поэлементное возведение в степень матрицы-произведения $NM$ с broadcasting вектора $p$ по строкам; во $g$ суммируются чётные элементы вектора $2p \odot q$.

In [2]:
Mp = (N @ M) ** p[:, None]        # возведение в степень с broadcasting
f1 = np.trace(Mp)
g1 = (2 * p * q)[0::2].sum()      # индикатор i%2==0 = срез по чётным
print('f1 =', f1)
print('g1 =', g1)

f1 = 1.9463327798987367e+22
g1 = 3960.962102855872


## Выражение 2

$f = 4 N_{k^*,59}$, $k^* = \arg\max_k q_k^2$;
$g = \sum_{i=24}^{69} \prod_{j=13}^{27} \left(7 \cdot 10^{-6} M^p\right)_{ij}^2$

In [3]:
k_star = np.argmax(q ** 2)
f2 = 4 * N[k_star, 59]
M_pow_p = M ** p[:, None]
block = (7e-6 * M_pow_p)[24:70, 13:28] ** 2
g2 = block.prod(axis=1).sum()
print('k* =', k_star)
print('f2 =', f2)
print('g2 =', g2)

k* = 5
f2 = 28.0
g2 = 4.010574190049982e-92


## Выражение 3

$f = \det(qq^T)^{p}$, $g = \left\| \{M_{ii} - N_{ii} + q^Tq\}_{i=0}^{84} \right\|_2$

Матрица $qq^T$ имеет ранг 1 при длине $q = 85$, поэтому её определитель равен нулю — проверяем это численно.

In [4]:
f3 = np.linalg.det(np.outer(q, q)) ** p[0]
vec3 = np.diag(M) - np.diag(N) + q @ q
g3 = np.linalg.norm(vec3, ord=2)
print('f3 =', f3)
print('g3 =', g3)

f3 = 0.0
g3 = 481328.2242846351


## Выражение 4

$f = q \sum_{i=0, i \neq 37}^{84} (N^p)_i$, $g = \sum_{i=12}^{76} I[i \neq M_{ii}](p \odot q)_i$

Трактовка: $N^p$ — поэлементная степень с broadcasting по строкам, $(N^p)_i$ — i-я строка; скалярные результаты: $f$ — скалярное произведение $q$ на вектор сумм строк (строка 37 исключена), $g$ — поэлементная сумма с индикатором (заметим: $M_{ii} = 3+2i > i$, поэтому индикатор тождественно равен 1 на всём диапазоне).

In [5]:
N_pow_p = N ** p[:, None]
row_sums = N_pow_p.sum(axis=1)
row_sums = row_sums * (np.arange(85) != 37)  # исключаем i = 37
f4 = q @ row_sums
idx = np.arange(12, 77)
mask = idx != M.diagonal()[idx]
g4 = (p * q)[idx][mask].sum()
print('f4 =', f4)
print('g4 =', g4)

f4 = 3273677100.7228413
g4 = 4905.871787500239


## Выражение 5

$f = p \cdot Tr(qv^T)$, $v = \{M_{84-i,84-i} + 2N_{i,i}\}_{i=0}^{84}$;
$g = \|q^T N\|_1$

$Tr(qv^T) = q \cdot v$ (след ранговой матрицы), далее скаляр умножается на сумму элементов $p$; $g$ — L1-норма вектора-произведения $q^T N$.

In [6]:
v = np.diag(M)[::-1] + 2 * np.diag(N)
f5 = p.sum() * np.trace(np.outer(q, v))
g5 = np.linalg.norm(q @ N, ord=1)
print('f5 =', f5)
print('g5 =', g5)

f5 = 52381990.735899836
g5 = 867476.0


## Итоговая таблица результатов

Все вычисления выполнены векторными средствами NumPy, без единого цикла.

In [7]:
results = {'f1': f1, 'g1': g1, 'f2': f2, 'g2': g2, 'f3': f3,
           'g3': g3, 'f4': f4, 'g4': g4, 'f5': f5, 'g5': g5}
summary = '\n'.join(f'{k} = {v:.6e}' for k, v in results.items())
print(summary)

f1 = 1.946333e+22
g1 = 3.960962e+03
f2 = 2.800000e+01
g2 = 4.010574e-92
f3 = 0.000000e+00
g3 = 4.813282e+05
f4 = 3.273677e+09
g4 = 4.905872e+03
f5 = 5.238199e+07
g5 = 8.674760e+05


## Выводы

В ходе задания освоены векторно-матричные вычисления в NumPy: матрицы заданной структуры строятся без циклов (векторизация `np.fill_diagonal`, `np.resize`, `np.where` с broadcasting масок), случайная выборка — через `default_rng`. Степенные выражения с вектором показателей реализованы через broadcasting, sums/products по диапазонам индексов — срезами (slicing), индикаторные функции — булевыми масками, след и нормы — функциями `np.trace`, `np.linalg.norm`. Все пять выражений вычислены за одну-две векторные операции каждое, что подтверждает эффективность векторного подхода против явных циклов.